In [1]:
import mne
import numpy as np

from pathlib import Path

from scipy.signal import butter
from scipy.signal import filtfilt
from scipy.signal import iirnotch
from scipy.signal import welch

from scipy.stats import skew
from scipy.stats import kurtosis

In [2]:
pairs = [
    ('FC5','FC6'),
    ('FC3','FC4'),
    ('FC1','FC2'),

    ('C5','C6'),
    ('C3','C4'),
    ('C1','C2'),

    ('CP5','CP6'),
    ('CP3','CP4'),
    ('CP1','CP2'),

    ('FP1','FP2'),

    ('AF7','AF8'),
    ('AF3','AF4'),

    ('F7','F8'),
    ('F5','F6'),
    ('F3','F4'),
    ('F1','F2'),

    ('FT7','FT8'),

    ('T7','T8'),
    ('T9','T10'),

    ('TP7','TP8'),

    ('P7','P8'),
    ('P5','P6'),
    ('P3','P4'),
    ('P1','P2'),

    ('PO7','PO8'),
    ('PO3','PO4'),

    ('O1','O2')
]

print("Number of pairs =", len(pairs))

Number of pairs = 27


In [3]:
def create_differential_channels(raw, pairs):

    # Clean channel names
    clean_names = {
        ch: ch.replace(".", "").upper()
        for ch in raw.ch_names
    }

    raw.rename_channels(clean_names)

    # Get EEG data
    data = raw.get_data()

    # Create fresh channel index dictionary
    ch_idx = {
        ch: i
        for i, ch in enumerate(raw.ch_names)
    }

    diff_data = []

    for left, right in pairs:

        left_idx = ch_idx[left]
        right_idx = ch_idx[right]

        diff_signal = data[left_idx] - data[right_idx]

        diff_data.append(diff_signal)

    diff_data = np.array(diff_data)

    diff_names = [
        f"{left}-{right}"
        for left, right in pairs
    ]

    return diff_data, diff_names

In [4]:
def apply_notch_filter(data, fs=160, freq=50, Q=30):

    b, a = iirnotch(
        w0=freq,
        Q=Q,
        fs=fs
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=1
    )

    return filtered

In [5]:
def apply_bandpass_filter(
    data,
    fs=160,
    lowcut=0.5,
    highcut=70,
    order=4
):

    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=1
    )

    return filtered

In [6]:
def extract_11_features(signal, fs=160):

    features = []

    # 1 Mean
    features.append(np.mean(signal))

    # 2 Variance
    features.append(np.var(signal))

    # 3 Skewness
    features.append(skew(signal))

    # 4 Kurtosis
    features.append(kurtosis(signal))

    # 5 Zero Crossing Count
    zc = np.sum(
        np.diff(
            np.sign(signal)
        ) != 0
    )

    features.append(zc)

    # 6 Area
    features.append(
        np.trapezoid(np.abs(signal))
    )

    # 7 Range
    features.append(
        np.max(signal) - np.min(signal)
    )

    # PSD
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=len(signal)
    )

    # 8 Delta
    delta = np.sum(
        psd[
            (freqs >= 0.5) &
            (freqs < 4)
        ]
    )

    # 9 Theta
    theta = np.sum(
        psd[
            (freqs >= 4) &
            (freqs < 8)
        ]
    )

    # 10 Alpha
    alpha = np.sum(
        psd[
            (freqs >= 8) &
            (freqs < 12)
        ]
    )

    # 11 Beta
    beta = np.sum(
        psd[
            (freqs >= 12) &
            (freqs < 30)
        ]
    )

    features.extend([
        delta,
        theta,
        alpha,
        beta
    ])

    return np.array(features)

In [7]:
def process_edf_file(edf_path):

    # --------------------------------------------------
    # 1. Load EDF
    # --------------------------------------------------
    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False
    )

    # --------------------------------------------------
    # 2. Extract Events
    # --------------------------------------------------
    events, event_id = mne.events_from_annotations(
    raw,
    verbose=False)

    movement_events = events[
        np.isin(
            events[:, 2],
            [event_id["T1"], event_id["T2"]]
        )
    ]

    # --------------------------------------------------
    # 3. Differential Channels
    # --------------------------------------------------
    diff_data, _ = create_differential_channels(
        raw,
        pairs
    )

    # --------------------------------------------------
    # 4. Filtering
    # --------------------------------------------------
    notch_data = apply_notch_filter(
        diff_data
    )

    filtered_data = apply_bandpass_filter(
        notch_data
    )
    # --------------------------------------------------
    # 5. No Normalization
    # --------------------------------------------------
    preprocessed_data = filtered_data

    # --------------------------------------------------
    # 6. Create 2-second Segments
    # --------------------------------------------------
    segment_length = 320

    segments = []
    labels = []

    for event in movement_events:

        start = event[0]
        end = start + segment_length

        if end <= preprocessed_data.shape[1]:

            segment = preprocessed_data[
                :,
                start:end
            ]

            segments.append(segment)

            labels.append(event[2])

    segments = np.array(segments)
    labels = np.array(labels)

    # --------------------------------------------------
    # 7. Convert Labels
    # T1 -> 0
    # T2 -> 1
    # --------------------------------------------------
    labels = np.where(
        labels == event_id["T1"],
        0,
        1
    )

    # --------------------------------------------------
    # 8. Create 7 Overlapping Windows
    # --------------------------------------------------
    window_size = 80
    step_size = 40

    all_windows = []

    for segment in segments:

        segment_windows = []

        for start in range(
            0,
            320 - window_size + 1,
            step_size
        ):

            end = start + window_size

            window = segment[
                :,
                start:end
            ]

            segment_windows.append(
                window
            )

        all_windows.append(
            segment_windows
        )

    all_windows = np.array(
        all_windows
    )

    # --------------------------------------------------
    # 9. Feature Extraction
    # --------------------------------------------------
    all_features = []

    for trial in all_windows:

        trial_features = []

        for window in trial:

            window_features = []

            for channel in window:

                feats = extract_11_features(
                    channel
                )

                window_features.extend(
                    feats
                )

            trial_features.append(
                window_features
            )

        all_features.append(
            trial_features
        )

    all_features = np.array(
        all_features
    )

    # --------------------------------------------------
    # Return
    # --------------------------------------------------
    return all_features, labels

In [8]:
excluded_subjects = [
    43,
    88,
    89,
    92,
    100,
    104
]

valid_subjects = []

for s in range(1,110):

    if s not in excluded_subjects:
        valid_subjects.append(s)

print(len(valid_subjects))

103


In [9]:
all_X = []
all_y = []
all_subjects = []

for subject in valid_subjects:

    subject_id = f"S{subject:03d}"

    for run in ["R04","R08","R12"]:

        edf_path = Path(
            f"../data/eegmmidb/files/{subject_id}/{subject_id}{run}.edf"
        )

        if not edf_path.exists():
            continue

        X_file, y_file = process_edf_file(
            edf_path
        )

        all_X.append(X_file)
        all_y.append(y_file)

        all_subjects.extend(
            [subject] * len(y_file)
        )

        print(
            f"{subject_id}{run} -> {X_file.shape}"
        )

S001R04 -> (15, 7, 297)
S001R08 -> (15, 7, 297)
S001R12 -> (15, 7, 297)
S002R04 -> (15, 7, 297)
S002R08 -> (15, 7, 297)
S002R12 -> (15, 7, 297)
S003R04 -> (15, 7, 297)
S003R08 -> (15, 7, 297)
S003R12 -> (15, 7, 297)
S004R04 -> (15, 7, 297)
S004R08 -> (15, 7, 297)
S004R12 -> (15, 7, 297)
S005R04 -> (15, 7, 297)
S005R08 -> (15, 7, 297)
S005R12 -> (15, 7, 297)
S006R04 -> (15, 7, 297)
S006R08 -> (15, 7, 297)
S006R12 -> (15, 7, 297)
S007R04 -> (15, 7, 297)
S007R08 -> (15, 7, 297)
S007R12 -> (15, 7, 297)
S008R04 -> (15, 7, 297)
S008R08 -> (15, 7, 297)
S008R12 -> (15, 7, 297)
S009R04 -> (15, 7, 297)
S009R08 -> (15, 7, 297)
S009R12 -> (15, 7, 297)
S010R04 -> (15, 7, 297)
S010R08 -> (15, 7, 297)
S010R12 -> (15, 7, 297)
S011R04 -> (15, 7, 297)
S011R08 -> (15, 7, 297)
S011R12 -> (15, 7, 297)
S012R04 -> (15, 7, 297)
S012R08 -> (15, 7, 297)
S012R12 -> (15, 7, 297)
S013R04 -> (15, 7, 297)
S013R08 -> (15, 7, 297)
S013R12 -> (15, 7, 297)
S014R04 -> (15, 7, 297)
S014R08 -> (15, 7, 297)
S014R12 -> (15, 

In [10]:
X = np.concatenate(
    all_X,
    axis=0
)

y = np.concatenate(
    all_y,
    axis=0
)

subjects = np.array(
    all_subjects
)

print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 7, 297)
(4635,)
(4635,)


In [11]:
print("Min:", X.min())
print("Max:", X.max())
print("Mean:", X.mean())
print("Std:", X.std())

Min: -5.2955442602654115
Max: 60.0
Mean: 1.6044713427336916
Std: 5.812257170418249


In [12]:
np.save(
    "../processed/X_no_norm.npy",
    X
)

np.save(
    "../processed/y_no_norm.npy",
    y
)

np.save(
    "../processed/subjects_no_norm.npy",
    subjects
)

print("Saved Successfully")

Saved Successfully


In [13]:
X = np.load("../processed/X_no_norm.npy")

print(X.shape)

(4635, 7, 297)


In [14]:
sample = X[0,0]

print(sample.shape)

(297,)


In [15]:
sample = sample.reshape(27,11)

print(sample)

[[-5.52267947e-06  5.09748287e-10  4.93488080e-01  6.65768392e-01
   1.30000000e+01  1.45872735e-03  1.19582921e-04  7.30375291e-11
   1.09611549e-10  4.83738433e-11  9.98373936e-11]
 [-1.69196582e-06  1.98793263e-10  5.96278007e-02 -3.12045409e-01
   1.70000000e+01  8.98539888e-04  6.49745664e-05  7.01637000e-12
   1.04223693e-11  2.06383593e-11  7.40281222e-11]
 [ 2.06049461e-06  6.56944502e-11 -9.21811951e-02 -2.75999965e-01
   1.40000000e+01  5.45979857e-04  3.91541038e-05  3.24834479e-12
   7.64130124e-12  9.45883815e-12  1.92999201e-11]
 [ 1.02405502e-06  6.37659038e-10  1.78245211e-01 -6.20992510e-01
   1.70000000e+01  1.66289411e-03  1.19217521e-04  3.08847891e-12
   2.72891017e-11  5.05373628e-11  1.02696982e-10]
 [-1.05813974e-06  2.33157690e-10  5.25275923e-02 -8.44830649e-01
   1.30000000e+01  1.04240219e-03  6.30389835e-05  8.85148027e-12
   2.00949435e-11  2.68382167e-11  3.77969987e-11]
 [-6.17763739e-06  1.77785164e-10  9.89331875e-03 -1.76331711e-01
   1.20000000e+01  

In [16]:
print(sample[:,7:11])

[[7.30375291e-11 1.09611549e-10 4.83738433e-11 9.98373936e-11]
 [7.01637000e-12 1.04223693e-11 2.06383593e-11 7.40281222e-11]
 [3.24834479e-12 7.64130124e-12 9.45883815e-12 1.92999201e-11]
 [3.08847891e-12 2.72891017e-11 5.05373628e-11 1.02696982e-10]
 [8.85148027e-12 2.00949435e-11 2.68382167e-11 3.77969987e-11]
 [1.95507248e-11 9.78097503e-12 3.40169345e-12 8.85773889e-12]
 [2.52569604e-11 5.47625080e-11 3.45295195e-11 4.88914207e-11]
 [1.34369317e-11 3.71591715e-11 2.18453805e-11 2.85196538e-11]
 [1.19808767e-12 9.96352309e-12 9.09501992e-12 8.92175991e-12]
 [9.36165388e-12 4.26899881e-12 5.66970873e-12 3.63745997e-11]
 [1.54435492e-10 1.46280611e-10 4.92816541e-11 1.37466088e-10]
 [2.99830469e-11 5.46591912e-11 1.32782953e-11 1.03371328e-10]
 [4.31873502e-10 4.98091533e-10 8.87955169e-11 1.85912289e-10]
 [3.72292804e-11 6.43373281e-11 1.77634471e-11 1.23093092e-10]
 [3.07140986e-12 3.34583458e-12 1.06191843e-12 1.61276285e-12]
 [1.75674225e-12 3.17993120e-12 1.84981516e-12 2.041449

In [17]:
sample = X[0,0].reshape(27,11)

print("Mean")
print(sample[:,0])

print("Variance")
print(sample[:,1])

print("Area")
print(sample[:,5])

print("Range")
print(sample[:,6])

print("Delta")
print(sample[:,7])

print("Theta")
print(sample[:,8])

print("Alpha")
print(sample[:,9])

print("Beta")
print(sample[:,10])

Mean
[-5.52267947e-06 -1.69196582e-06  2.06049461e-06  1.02405502e-06
 -1.05813974e-06 -6.17763739e-06 -2.86201008e-06 -1.66673587e-06
  1.20358309e-06 -5.42693415e-06 -4.72068661e-08 -4.70625401e-06
 -6.06766817e-06  3.27642083e-07 -5.27601898e-06  3.94309497e-07
 -2.44095423e-06 -4.38967329e-06 -5.65259009e-06 -4.82870141e-06
 -3.71767112e-06 -8.21668413e-07 -7.43608974e-07  3.10756950e-07
 -1.18722097e-06 -3.91237236e-06  2.47089657e-06]
Variance
[5.09748287e-10 1.98793263e-10 6.56944502e-11 6.37659038e-10
 2.33157690e-10 1.77785164e-10 4.56549541e-10 2.51628142e-10
 8.27286723e-11 1.67951088e-10 7.55831551e-10 3.11604576e-10
 1.83854836e-09 3.81652449e-10 2.11977342e-11 5.84188981e-11
 1.05413310e-09 1.62294443e-09 9.29390195e-10 7.92717681e-10
 5.89356454e-10 4.45518136e-10 3.28014800e-10 1.36262768e-10
 6.51515365e-10 5.07647215e-10 6.50911418e-10]
Area
[0.00145873 0.00089854 0.00054598 0.00166289 0.0010424  0.00092666
 0.0013843  0.00105706 0.00059878 0.0008942  0.00173663 0.001

In [18]:
import mne
raw = mne.io.read_raw_edf(
    "../data/eegmmidb/files/S001/S001R04.edf",
    preload=True,
    verbose=False
)

print(raw.get_data().min())
print(raw.get_data().max())

-0.000376
0.0005949999999999999
